# Readout IQ blob classification

Feed a dense cloud of single-shot IQ values to the VLM. The model recognizes
under-separated blobs, leakage to `|2⟩`, and skewed populations — all things
that degrade readout fidelity. We also sketch a cheap linear threshold for
reference.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 4000
ground = rng.normal(loc=(-0.9, 0.1), scale=(0.35, 0.35), size=(n//2, 2))
excited = rng.normal(loc=(+0.8, 0.4), scale=(0.40, 0.40), size=(n//2, 2))
iq = np.vstack([ground, excited])
np.save('readout_iq.npy', iq)
iq.shape

In [ ]:
from qcal.data import from_npy

payload = from_npy('readout_iq.npy', experiment_type='readout_iq')
payload.image

In [ ]:
from qcal.analyzer import analyze_payload

result = analyze_payload(payload, backend='auto')
result

Once the VLM returns a `recommended_parameters.readout_threshold`, wire it into
your control stack's classifier. Below: a crude linear threshold along the I-axis
for reference.

In [ ]:
import matplotlib.pyplot as plt

thresh = result.parsed.get('recommended_parameters', {}).get('readout_threshold', 0.0)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(iq[:, 0], iq[:, 1], s=3, alpha=0.4)
ax.axvline(thresh, color='r', linewidth=1.0, label=f'threshold = {thresh}')
ax.set_xlabel('I'); ax.set_ylabel('Q'); ax.legend(); ax.grid(alpha=0.3)